# Train Sentiment Model

Run this notebook on a cloud GPU to create the sentiment model artifacts. The final model is saved to `src/model`, which is the directory the backend loads at runtime.

Your training CSV must contain two columns: `text` and `label`. Labels may be `-1`, `0`, `1`, `negative`, `neutral`, `positive`, `bearish`, or `bullish`.

In [ ]:
# Run this in Colab/Kaggle/cloud notebooks if the packages are not already installed.
%pip install -q transformers datasets accelerate evaluate scikit-learn pandas torch

In [ ]:
from pathlib import Path

BASE_MODEL = "ProsusAI/finbert"
TRAINING_CSV = "../src/data/Reddit_Data.csv"
MODEL_DIR = "../src/model"
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

If you are running this outside the repo, upload your CSV and change `TRAINING_CSV` above to that uploaded file path.

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

def normalize_training_dataframe(df):
    required_columns = {"text", "label"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Training CSV missing required columns: {sorted(missing_columns)}")

    df = df[["text", "label"]].copy()
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    df = df[df["text"] != ""]
    df = df.drop_duplicates(subset=["text"])

    label_map = {
        -1: 0,
        0: 1,
        1: 2,
        "-1": 0,
        "0": 1,
        "1": 2,
        "negative": 0,
        "neutral": 1,
        "positive": 2,
        "bearish": 0,
        "bullish": 2,
    }
    df["label"] = df["label"].map(lambda value: label_map.get(str(value).strip().lower(), label_map.get(value)))

    invalid_count = df["label"].isna().sum()
    if invalid_count:
        raise ValueError(f"Training CSV contains {invalid_count} rows with unsupported labels.")

    df["label"] = df["label"].astype(int)
    label_counts = df["label"].value_counts()
    if len(label_counts) < 2:
        raise ValueError("Training CSV must contain at least two sentiment classes.")
    if label_counts.min() < 2:
        raise ValueError("Each sentiment class must have at least two rows for stratified splitting.")

    return df

df = normalize_training_dataframe(pd.read_csv(TRAINING_CSV))
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42,
)

dataset_train = Dataset.from_pandas(train_df, preserve_index=False)
dataset_test = Dataset.from_pandas(test_df, preserve_index=False)
df["label"].value_counts().sort_index()

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    texts = [str(text) if isinstance(text, str) else "" for text in batch["text"]]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=512)

dataset_train = dataset_train.map(tokenize, batched=True)
dataset_test = dataset_test.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

In [ ]:
import os

os.environ["TENSORBOARD_LOGGING_DIR"] = f"{MODEL_DIR}/logs"

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
metrics = trainer.evaluate()
metrics

In [ ]:
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

model_files = sorted(str(path) for path in Path(MODEL_DIR).glob("*"))
model_files

In [ ]:
# Optional: create a zip you can download from the cloud notebook and unpack into this repo's src/model folder.
import shutil

archive_path = shutil.make_archive("sentiment_model", "zip", MODEL_DIR)
archive_path